# Simple mujoco control
- UR5e
- Mujoco Python viewer
- Control with actuator name & index

In [1]:
import os
import sys
sys.path.append(os.path.abspath('../'))
# from pp_base_mujoco.VIEWER import MUJOCOGLVIEWER
from pp_base_mujoco.VIEWER import *

import mujoco
import numpy as np
import time

import os

In [2]:
xml_path = '../asset/ur_scene.xml'
xml_abs_path = os.path.abspath(xml_path)

model = mujoco.MjModel.from_xml_path(xml_abs_path)
data = mujoco.MjData(model)

### 1. Qpos - kinematics control
- use model.joint.qposadr[0]
    - this qposadr[0] considers free & ball joints

In [5]:
number_of_q = model.nq # dof
number_of_joint = model.njnt

joint_enum = mujoco.mjtObj.mjOBJ_JOINT
joint_names = [mujoco.mj_id2name(model,mujoco.mjtObj.mjOBJ_JOINT,joint_idx) for joint_idx in range(model.njnt)]
joint_qpos_idx = [model.joint(joint_name).qposadr[0] for joint_name in joint_names]

print("joint names:", joint_names)
print("joint qpos index:", joint_qpos_idx)

joint names: ['shoulder_pan_joint', 'shoulder_lift_joint', 'elbow_joint', 'wrist_1_joint', 'wrist_2_joint', 'wrist_3_joint']
joint qpos index: [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5)]


In [ ]:
""" FUNCTIONS: APPLY QPOS """

def get_joint_names (model, data):
    joint_names = [mujoco.mj_id2name(model,mujoco.mjtObj.mjOBJ_JOINT,joint_idx) for joint_idx in range(model.njnt)]
    return joint_names

def apply_qpos_idxs (model, data, idxs, value):
    if len(idxs) != len(value):
        raise ValueError("length of name and value is different")
    qpos_ = np.zeros(model.nq) # number of qpos
    for i, idx in enumerate(idxs):
        qpos_[idx] = value[i]
    data.qpos = qpos_    
    return None

def apply_qpos_names (model, data, names, value):
    if len(names) != len(value):
        raise ValueError("length of names and value is different")
    # initialize
    indexs = [model.joint(joint_name).qposadr[0] for joint_name in names]
    qpos_ = np.zeros(model.nq) # number of qpos

    for i, idx in enumerate(indexs):
        qpos_[idx] = value[i]
    data.qpos = qpos_
    return None

### 2. Actuator - torque control

In [4]:
# number of control
number_of_control = model.nu
actuator_enum = mujoco.mjtObj.mjOBJ_ACTUATOR
control_names = [mujoco.mj_id2name(model,mujoco.mjtObj.mjOBJ_ACTUATOR,ctrl_idx) for ctrl_idx in range(model.nu)]
print("Actuator names:", control_names)

# get controller range
control_range = model.actuator_ctrlrange
print("Control range:", control_range)

Actuator names: ['shoulder_pan', 'shoulder_lift', 'elbow', 'wrist_1', 'wrist_2', 'wrist_3']
Control range: [[-6.2831  6.2831]
 [-6.2831  6.2831]
 [-3.1415  3.1415]
 [-6.2831  6.2831]
 [-6.2831  6.2831]
 [-6.2831  6.2831]]


In [ ]:
""" FUNCTIONS: APPLY ACTUATOR FORCE """

def get_actuator_names (model, data):
    control_names = [mujoco.mj_id2name(model,mujoco.mjtObj.mjOBJ_ACTUATOR,ctrl_idx) for ctrl_idx in range(model.nu)]
    return control_names

def apply_control_idxs (model, data, idxs, value):
    if len(idxs) != len(value):
        raise ValueError("length of name and value is different")
    ctrl_ = np.zeros(model.nu) # number of control
    for i, idx in enumerate(idxs):
        ctrl_[idx] = value[i]
    data.ctrl = ctrl_    
    return None

def apply_ctrl_names (model, data, names, value):
    if len(names) != len(value):
        raise ValueError("length of names and value is different")
    # initialize
    control_names = [mujoco.mj_id2name(model,mujoco.mjtObj.mjOBJ_ACTUATOR,ctrl_idx) for ctrl_idx in range(model.nu)]
    ctrl_ = np.zeros(model.nu) # number of control

    for i, n in enumerate(names):
        if n in control_names:
            idx = control_names.index(n)
            ctrl_[idx] = value[i]
        else:
            print(f"Name {n} is not included in actuator names, passing..")
    data.ctrl = ctrl_
    return None

In [5]:
""" MAIN LOOP """
# create python viewer object
viewer = MUJOCOGLVIEWER(model, data)
# data reset
mujoco.mj_resetData(model, data)

# get actuator names
actuator_names = get_actuator_names(model, data)
print("Actuator names:", actuator_names)

while True:
    if viewer.is_alive():

        # apply control 
        apply_ctrl_name(model, data, name=["shoulder_lift", "elbow"], value=[0.8, 0.8]) # now data 
        mujoco.mj_step(model, data)
        viewer.render()

    else:
        break

# close
viewer.close()

Actuator names: ['shoulder_pan', 'shoulder_lift', 'elbow', 'wrist_1', 'wrist_2', 'wrist_3']
